# Clover Dirac Operator Test

Sheikholeslami-Wohlert improved Wilson Dirac operator:

$$D_{\text{cl}} = D_W + \frac{i\, c_{SW}}{4} \sum_{\mu < \nu} \sigma_{\mu\nu} \hat{F}_{\mu\nu}$$

### Tests:
1. $\hat{F}_{\mu\nu}$가 anti-hermitian인지 확인
2. $\hat{F}_{\mu\nu}$가 traceless인지 확인
3. `CloverDiracOperator` 생성 및 적용
4. $c_{SW}=0$이면 Wilson과 동일한지 확인
5. $c_{SW}\neq 0$이면 Wilson과 다른지 확인
6. free field ($U=I$)에서 clover term이 0인지 확인

In [1]:
using LatticeDiracOperators
using Gaugefields
using LMP
using LinearAlgebra
using StaticArrays

NX = 4
NY = 4
NZ = 4
NT = 4
Nwing = 0
NC = 3

params = Dict()
params["Dirac_operator"] = "Wilson"
params["κ"] = 0.16
# params["faster version"] = true
params["eps_CG"] = 1.0e-8
params["verbose_level"] = 1

println("Setup done.")

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   SpecialFunctionsExt [1285c0f1-ff9f-5867-b66e-0f359bcf09ba]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   CUDA [052768ef-5323-5732-b1bb-66c8b64840ba]
│   SparseArraysExt [85068d23-b5fb-53f1-8204-05c2aba6942f]
│   StructArraysGPUArraysCoreExt [cbbf52aa-d660-55d4-8a5a-33845bbaa85d]
│   AtomixCUDAExt [13011619-4c7c-5ef0-948f-5fc81565cd05]
│   LinearAlgebraExt [66d79d19-2cc4-5b0b-ac7a-b340256d1ecd]
│   ZygoteColorsExt [e68c091a-8ea5-5ca7-be4f-380657d4ad79]
│   ChainRulesCoreExt [eae2faf6-b232-58cb-a410-7764fda2830c]
│   Zygote [e88e6eb3-aa80-5325-afca-941959d7151f]
└ @ Pkg.API C:\Users\_s2220186\AppData\Local\Programs\Julia-1.10.7\share\julia\stdlib\v1.10\Pkg\src\API.jl:1279
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ 

Setup done.


## Test 1: Free field ($U = I$) — Clover term should vanish

In [2]:
# Free field: 모든 링크 = 단위행렬
U_free = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="cold")

# F_μν가 전부 0인지 확인
println("=== Free field F̂_μν test ===")
all_zero = true
for mu = 1:4, nu = (mu+1):4
    Fmn = compute_Fmunu(U_free, mu, nu)
    max_val = maximum(abs.(Fmn))
    status = max_val < 1e-14 ? "PASS ✓" : "FAIL ✗"
    println("  F̂_($mu,$nu): max|F| = $(max_val)  $status")
    if max_val >= 1e-14
        all_zero = false
    end
end
println(all_zero ? "=> Free field test PASSED" : "=> Free field test FAILED")

=== Free field F̂_μν test ===
  F̂_(1,2): max|F| = 0.0  PASS ✓
  F̂_(1,3): max|F| = 0.0  PASS ✓
  F̂_(1,4): max|F| = 0.0  PASS ✓
  F̂_(2,3): max|F| = 0.0  PASS ✓
  F̂_(2,4): max|F| = 0.0  PASS ✓
  F̂_(3,4): max|F| = 0.0  PASS ✓
=> Free field test PASSED


## Test 2: Hot field — $\hat{F}_{\mu\nu}$ anti-hermitian & traceless

In [3]:
# Hot (random) gauge field
U_hot = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")

println("=== Anti-hermiticity & tracelessness test ===")
max_antiherm = 0.0
max_trace = 0.0

for mu = 1:4, nu = (mu+1):4
    Fmn = compute_Fmunu(U_hot, mu, nu)
    
    for it = 1:NT, iz = 1:NZ, iy = 1:NY, ix = 1:NX
        # 3×3 행렬 추출
        F = SMatrix{3,3,ComplexF64,9}(
            Fmn[1,1,ix,iy,iz,it], Fmn[2,1,ix,iy,iz,it], Fmn[3,1,ix,iy,iz,it],
            Fmn[1,2,ix,iy,iz,it], Fmn[2,2,ix,iy,iz,it], Fmn[3,2,ix,iy,iz,it],
            Fmn[1,3,ix,iy,iz,it], Fmn[2,3,ix,iy,iz,it], Fmn[3,3,ix,iy,iz,it]
        )
        
        # Anti-hermitian check: F† = -F
        dev_ah = maximum(abs.(F + F'))
        max_antiherm = max(max_antiherm, dev_ah)
        
        # Traceless check: tr(F) = 0
        dev_tr = abs(tr(F))
        max_trace = max(max_trace, dev_tr)
    end
end

println("  max|F + F†| = $(max_antiherm)  $(max_antiherm < 1e-13 ? "PASS ✓" : "FAIL ✗")")
println("  max|tr(F)|  = $(max_trace)  $(max_trace < 1e-13 ? "PASS ✓" : "FAIL ✗")")

=== Anti-hermiticity & tracelessness test ===
  max|F + F†| = 0.0  PASS ✓
  max|tr(F)|  = 2.220446049250313e-16  PASS ✓


## Test 3: $c_{SW}=0$ ⟹ Clover = Wilson

In [4]:
x = Initialize_pseudofermion_fields(U_hot[1], "Wilson", nowing=true)
gauss_distribution_fermion!(x)

# Wilson Dirac operator
D_W = Dirac_operator(U_hot, x, params)
y_wilson = D_W * x

# Clover with c_SW = 0 (should be identical to Wilson)
D_cl0 = CloverDiracOperator(D_W, U_hot, 0.0)
y_cl0 = D_cl0 * x

diff_0 = sqrt(real(dot(y_cl0 - y_wilson, y_cl0 - y_wilson)))
println("=== c_SW = 0 test ===")
println("  ||D_cl(c_SW=0)*x - D_W*x|| = $(diff_0)")
println("  $(diff_0 < 1e-14 ? "PASS ✓" : "FAIL ✗")")

=== c_SW = 0 test ===
  ||D_cl(c_SW=0)*x - D_W*x|| = 0.0
  PASS ✓


## Test 4: $c_{SW} \neq 0$ ⟹ Clover ≠ Wilson (hot field)

In [5]:
# Clover with c_SW = 1.0
D_cl1 = CloverDiracOperator(D_W, U_hot, 1.0)
y_cl1 = D_cl1 * x

diff_1 = sqrt(real(dot(y_cl1 - y_wilson, y_cl1 - y_wilson)))
norm_w = sqrt(real(dot(y_wilson, y_wilson)))

println("=== c_SW = 1.0 test ===")
println("  ||D_cl*x - D_W*x|| = $(diff_1)")
println("  ||D_W*x||          = $(norm_w)")
println("  relative diff      = $(diff_1/norm_w)")
println("  $(diff_1 > 1e-10 ? "PASS ✓ (Clover ≠ Wilson)" : "FAIL ✗ (should differ)")")

=== c_SW = 1.0 test ===
  ||D_cl*x - D_W*x|| = 7.026959866984445
  ||D_W*x||          = 64.82462794824305
  relative diff      = 0.10839954025798458
  PASS ✓ (Clover ≠ Wilson)


## Test 5: Free field에서 Clover = Wilson

In [6]:
x_free = Initialize_pseudofermion_fields(U_free[1], "Wilson", nowing=true)
gauss_distribution_fermion!(x_free)

D_W_free = Dirac_operator(U_free, x_free, params)
y_w_free = D_W_free * x_free

# c_SW = 1.0이어도 free field이면 F=0이므로 Wilson과 같아야 함
D_cl_free = CloverDiracOperator(D_W_free, U_free, 1.0)
y_cl_free = D_cl_free * x_free

diff_free = sqrt(real(dot(y_cl_free - y_w_free, y_cl_free - y_w_free)))
println("=== Free field Clover = Wilson test ===")
println("  ||D_cl(free)*x - D_W(free)*x|| = $(diff_free)")
println("  $(diff_free < 1e-14 ? "PASS ✓" : "FAIL ✗")")

=== Free field Clover = Wilson test ===
  ||D_cl(free)*x - D_W(free)*x|| = 0.0
  PASS ✓


## Test 6: $c_{SW}$ linearity — $D_{cl}(2c_{SW}) - D_W = 2(D_{cl}(c_{SW}) - D_W)$

In [7]:
# Clover term은 c_SW에 linear
D_cl2 = CloverDiracOperator(D_W, U_hot, 2.0)
y_cl2 = D_cl2 * x

# Clover part: D_cl - D_W
# clover_1 = y_cl1 - y_wilson  (c_SW=1)
# clover_2 = y_cl2 - y_wilson  (c_SW=2)
# clover_2 ≈ 2 * clover_1 이어야 함

clover_part_1 = y_cl1 - y_wilson
clover_part_2 = y_cl2 - y_wilson

# 2*clover_1 계산
two_clover_1 = deepcopy(clover_part_1)
axpby!(2.0, clover_part_1, 0.0, two_clover_1)

diff_lin = two_clover_1 - clover_part_2
err_lin = sqrt(real(dot(diff_lin, diff_lin)))
norm_cl = sqrt(real(dot(clover_part_2, clover_part_2)))

println("=== c_SW linearity test ===")
println("  ||2·clover(c=1) - clover(c=2)|| = $(err_lin)")
println("  ||clover(c=2)||                  = $(norm_cl)")
println("  relative error                   = $(err_lin / norm_cl)")
println("  $(err_lin/norm_cl < 1e-13 ? "PASS ✓" : "FAIL ✗")")

=== c_SW linearity test ===
  ||2·clover(c=1) - clover(c=2)|| = 7.401028921952105e-15
  ||clover(c=2)||                  = 14.053919733968886
  relative error                   = 5.26616706374345e-16
  PASS ✓


## Test 7: `refresh_clover!` 동작 확인

In [8]:
# D_cl1은 U_hot으로 만든 것 — U_free로 교체 후 refresh
D_cl_refresh = CloverDiracOperator(D_W, U_hot, 1.0)
y_before = D_cl_refresh * x  # hot 게이지 적용

# 게이지를 free로 교체
D_cl_refresh.U = U_free
refresh_clover!(D_cl_refresh)

# refresh 후 clover term = 0이어야 함 → Wilson part만 남음
# (단, D_W는 여전히 U_hot이므로 Wilson part 자체는 변하지 않음)
y_after = D_cl_refresh * x

# y_after에서 clover part는 0, Wilson part는 y_wilson과 동일
diff_refresh = sqrt(real(dot(y_after - y_wilson, y_after - y_wilson)))
println("=== refresh_clover! test ===")
println("  ||D_cl(refreshed free)*x - D_W*x|| = $(diff_refresh)")
println("  $(diff_refresh < 1e-14 ? "PASS ✓" : "FAIL ✗")")

=== refresh_clover! test ===
  ||D_cl(refreshed free)*x - D_W*x|| = 0.0
  PASS ✓


## Test 8: σ_μν 성질 확인 (LatticeDiracOperators γ convention)

$\sigma_{\mu\nu} = i\gamma_\mu \gamma_\nu$, Euclidean: $\{\gamma_\mu,\gamma_\nu\}=2\delta_{\mu\nu}$

- hermitian: $\sigma_{\mu\nu}^\dagger = \sigma_{\mu\nu}$
- antisymmetric: $\sigma_{\mu\nu} = -\sigma_{\nu\mu}$

In [9]:
# LatticeDiracOperators의 Euclidean γ 행렬 (mk_gamma 함수와 동일)
g = Vector{Matrix{ComplexF64}}(undef, 4)
g[1] = ComplexF64[0 0 0 -im;  0 0 -im 0;  0 im 0 0;  im 0 0 0]
g[2] = ComplexF64[0 0 0 -1;   0 0 1 0;    0 1 0 0;   -1 0 0 0]
g[3] = ComplexF64[0 0 -im 0;  0 0 0 im;   im 0 0 0;  0 -im 0 0]
g[4] = ComplexF64[0 0 -1 0;   0 0 0 -1;   -1 0 0 0;  0 -1 0 0]

println("=== Euclidean γ anti-commutation: {γ_μ,γ_ν} = 2δ_μν ===")
clifford_ok = true
for mu = 1:4, nu = mu:4
    anticomm = g[mu]*g[nu] + g[nu]*g[mu]
    expected = 2 * (mu == nu ? I(4) : zeros(4,4))
    err = maximum(abs.(anticomm - expected))
    ok = err < 1e-15
    clifford_ok &= ok
    mu <= nu && println("  {γ_$mu,γ_$nu}: err=$(err)  $(ok ? "✓" : "✗")")
end
println(clifford_ok ? "=> Clifford algebra PASSED" : "=> Clifford algebra FAILED")

println("\n=== σ_μν properties ===")
all_ok = true
for mu = 1:4, nu = (mu+1):4
    sig = im * g[mu] * g[nu]
    
    # hermitian: σ† = σ  (Euclidean hermitian gammas → σ_μν hermitian)
    err_herm = maximum(abs.(sig - sig'))
    
    # antisymmetric: σ_μν = -σ_νμ
    sig_rev = im * g[nu] * g[mu]
    err_as = maximum(abs.(sig + sig_rev))
    
    ok = err_herm < 1e-15 && err_as < 1e-15
    all_ok &= ok
    println("  σ_($mu,$nu): hermitian=$(err_herm), antisym=$(err_as)  $(ok ? "✓" : "✗")")
end
println(all_ok ? "=> σ test PASSED" : "=> σ test FAILED")

=== Euclidean γ anti-commutation: {γ_μ,γ_ν} = 2δ_μν ===
  {γ_1,γ_1}: err=0.0  ✓
  {γ_1,γ_2}: err=0.0  ✓
  {γ_1,γ_3}: err=0.0  ✓
  {γ_1,γ_4}: err=0.0  ✓
  {γ_2,γ_2}: err=0.0  ✓
  {γ_2,γ_3}: err=0.0  ✓
  {γ_2,γ_4}: err=0.0  ✓
  {γ_3,γ_3}: err=0.0  ✓
  {γ_3,γ_4}: err=0.0  ✓
  {γ_4,γ_4}: err=0.0  ✓
=> Clifford algebra PASSED

=== σ_μν properties ===
  σ_(1,2): hermitian=0.0, antisym=0.0  ✓
  σ_(1,3): hermitian=0.0, antisym=0.0  ✓
  σ_(1,4): hermitian=0.0, antisym=0.0  ✓
  σ_(2,3): hermitian=0.0, antisym=0.0  ✓
  σ_(2,4): hermitian=0.0, antisym=0.0  ✓
  σ_(3,4): hermitian=0.0, antisym=0.0  ✓
=> σ test PASSED


## Summary

In [10]:
println("========================")
println("  Clover Test Summary")
println("========================")
println("1. Free field F=0:       $(all_zero ? "PASS" : "FAIL")")
println("2. Anti-herm & traceless: $(max_antiherm < 1e-13 && max_trace < 1e-13 ? "PASS" : "FAIL")")
println("3. c_SW=0 → Wilson:      $(diff_0 < 1e-14 ? "PASS" : "FAIL")")
println("4. c_SW≠0 → ≠Wilson:     $(diff_1 > 1e-10 ? "PASS" : "FAIL")")
println("5. Free Clover=Wilson:    $(diff_free < 1e-14 ? "PASS" : "FAIL")")
println("6. c_SW linearity:        $(err_lin/norm_cl < 1e-13 ? "PASS" : "FAIL")")
println("7. refresh_clover!:       $(diff_refresh < 1e-14 ? "PASS" : "FAIL")")
println("8. Clifford algebra:      $(clifford_ok ? "PASS" : "FAIL")")
println("9. σ_μν properties:       $(all_ok ? "PASS" : "FAIL")")

  Clover Test Summary
1. Free field F=0:       PASS
2. Anti-herm & traceless: PASS
3. c_SW=0 → Wilson:      PASS
4. c_SW≠0 → ≠Wilson:     PASS
5. Free Clover=Wilson:    PASS
6. c_SW linearity:        PASS
7. refresh_clover!:       PASS
8. Clifford algebra:      PASS
9. σ_μν properties:       PASS
